In [4]:
import pandas as pd
from deltalake import DeltaTable, write_deltalake

In [12]:

## Load delta lake from remote
table_path = "s3://delta-table-storage/stocks"
storage_options = {
    "AWS_ACCESS_KEY_ID": "CzOwnLkEDXQy951AOqes",
    "AWS_SECRET_ACCESS_KEY": "fdRe91TOtqTl0icUkZLsUnWvZa90aZ5qG5rVEf7S",
    "AWS_ENDPOINT_URL": "http://192.168.1.5:9000",
    "AWS_ALLOW_HTTP": "true",
    "AWS_EC2_METADATA_DISABLED": "true",
    "AWS_REGION": 'us-east-1',
    "aws_conditional_put": "etag",
}

dt = DeltaTable(table_uri=table_path, storage_options=storage_options)
df = dt.to_pandas()
print(df.head())

              key       date        open  ...  close  volume  symbol
0  KSV_2025-05-21 2025-05-21  192.300000  ...  188.0  118700     KSV
1  KSV_2025-08-13 2025-08-13  168.000000  ...  166.0   35700     KSV
2  KSV_2025-08-04 2025-08-04  167.899994  ...  167.5   16300     KSV
3  KSV_2025-05-19 2025-05-19  170.000000  ...  159.0   91000     KSV
4  KSV_2025-08-22 2025-08-22  161.500000  ...  157.0   51100     KSV

[5 rows x 8 columns]


In [13]:
## Sync deltalake from remote to local

## Sync deltalake from remote to local  
table_path = "s3://delta-table-storage/stocks"
storage_options = {
    "AWS_ACCESS_KEY_ID": "JzgMMlm2rZcHlIsV1UBd",
    "AWS_SECRET_ACCESS_KEY": "x872pkjyArcN1LoDjmkqxA4e51xxsJoDyourKaKf",
    "AWS_ENDPOINT_URL": "http://localhost:9000",
    "AWS_ALLOW_HTTP": "true",
    "AWS_EC2_METADATA_DISABLED": "true",
    "AWS_REGION": 'us-east-1',
    "aws_conditional_put": "etag",
}

result = write_deltalake(table_path, df, storage_options=storage_options, mode='overwrite')
print(result)

None


# Test

In [17]:
table_path = "s3://delta-table-storage/stocks"
storage_options = {
    "AWS_ACCESS_KEY_ID": "JzgMMlm2rZcHlIsV1UBd",
    "AWS_SECRET_ACCESS_KEY": "x872pkjyArcN1LoDjmkqxA4e51xxsJoDyourKaKf",
    "AWS_ENDPOINT_URL": "http://localhost:9000",
    "AWS_ALLOW_HTTP": "true",
    "AWS_EC2_METADATA_DISABLED": "true",
    "AWS_REGION": 'us-east-1',
    "aws_conditional_put": "etag",
}


dt = DeltaTable(table_uri=table_path, storage_options=storage_options)
df = dt.to_pandas(filters=[('symbol', '=', 'VNINDEX')])
df = df.sort_values(by='date', ascending=False)
print(df.head())

                     key       date  ...      volume   symbol
6071  VNINDEX_2025-08-29 2025-08-29  ...  1519909888  VNINDEX
6070  VNINDEX_2025-08-28 2025-08-28  ...  1184795776  VNINDEX
498   VNINDEX_2025-08-27 2025-08-27  ...  1553390592  VNINDEX
1995  VNINDEX_2025-08-26 2025-08-26  ...  1402350720  VNINDEX
3200  VNINDEX_2025-08-25 2025-08-25  ...  1501884288  VNINDEX

[5 rows x 8 columns]


In [21]:
## Drop schema from deltalake

table_path = "s3://delta-table-storage/stocks_feature_store"
storage_options = {
    "AWS_ACCESS_KEY_ID": "JzgMMlm2rZcHlIsV1UBd",
    "AWS_SECRET_ACCESS_KEY": "x872pkjyArcN1LoDjmkqxA4e51xxsJoDyourKaKf",
    "AWS_ENDPOINT_URL": "http://localhost:9000",
    "AWS_ALLOW_HTTP": "true",
    "AWS_EC2_METADATA_DISABLED": "true",
    "AWS_REGION": 'us-east-1',
    "aws_conditional_put": "etag",
}


In [23]:
dt = DeltaTable(table_uri=table_path, storage_options=storage_options)
df = dt.to_pandas(filters=[('symbol', '=', 'VIX')])
df = df.sort_values(by='date', ascending=False)
print(df.head())
print(df.tail())

          date symbol  ...             key  __index_level_0__
244 2025-08-29    VIX  ...  VIX_2025-08-29             178389
243 2025-08-28    VIX  ...  VIX_2025-08-28             178030
242 2025-08-27    VIX  ...  VIX_2025-08-27             177671
241 2025-08-26    VIX  ...  VIX_2025-08-26             177312
240 2025-08-25    VIX  ...  VIX_2025-08-25             176953

[5 rows x 58 columns]
        date symbol  close  ...  zscore_kf_20             key  __index_level_0__
4 2024-09-13    VIX  11.30  ...     -0.841132  VIX_2024-09-13              92229
3 2024-09-12    VIX  11.25  ...     -0.437286  VIX_2024-09-12              91870
2 2024-09-11    VIX  11.30  ...      0.031767  VIX_2024-09-11              91511
1 2024-09-10    VIX  11.35  ...      0.505858  VIX_2024-09-10              91152
0 2024-09-09    VIX  11.65  ...      1.008665  VIX_2024-09-09              90793

[5 rows x 58 columns]


# Sync data from local storage file

In [ ]:
from custom_metastock2pd import metastock_read, metastock_read_master, metastock_emaster, metastock_xmaster
import pandas as pd
FDATA_INDEX_DIR = "D:\\fdata_ami\\MetaStock\\EOD\\Chi so"
emaster_index_df = metastock_read_master(FDATA_INDEX_DIR)
from deltalake import DeltaTable, write_deltalake

df = pd.DataFrame()
for index, row in emaster_index_df.iterrows():
    try:
        # skip specific index symbols
        if row["symbol"] in ["VNINDEX", "VN30"]:
            continue
        print("Processing " , row["symbol"], " ...")
        tickDf = metastock_read(row["filename"], extra_buffer=50)
        tickDf = tickDf.sort_index().reset_index(names='date')
        tickDf['symbol'] = row['symbol']
        df = pd.concat([df, tickDf])
    except Exception as e:
        print(e)
        print("Cannot read file: ", row["filename"])

try:
    df['date'] = pd.to_datetime(df['date'], format='%Y%m%d')
except ValueError:
    # Fallback to automatic parsing if format doesn't match
    df['date'] = pd.to_datetime(df['date'])

df['key'] = df["symbol"] + "_" + df['date'].dt.strftime('%Y-%m-%d')
# Create a new column 'key' by concatenating 'symbol' and formatted 'date']

## select only the columns we need
df = df[["key", "symbol", "date", "open", "high", "low", "close", "volume"]]

storage_options = {
    "AWS_ACCESS_KEY_ID": "CzOwnLkEDXQy951AOqes",
    "AWS_SECRET_ACCESS_KEY": "fdRe91TOtqTl0icUkZLsUnWvZa90aZ5qG5rVEf7S",
    "AWS_ENDPOINT_URL": "http://localhost:9000",
    "AWS_ALLOW_HTTP": "true",
    "AWS_EC2_METADATA_DISABLED": "true",
    "AWS_REGION": 'ap-southeast-1',
    "aws_conditional_put": "etag",
}

# print(destination)
dt = DeltaTable("s3://delta-table-storage/stocks", storage_options=storage_options)
result = dt.merge(df, 
        predicate="target.key == source.key",
        source_alias="source",
        target_alias="target"
) \
    .when_not_matched_insert_all() \
    .when_matched_update_all(predicate="target.key == source.key " \
        "AND target.volume != source.volume AND target.close != source.close")\
    .execute()
print(result)

Processing  3700  ...
Processing  1300  ...
Processing  3785  ...
Processing  3763  ...
Processing  9578  ...
Processing  2727  ...
Processing  4530  ...
Processing  2300  ...
Processing  VNUTI  ...
Processing  VNX50  ...
Processing  0573  ...
Processing  VNFINLEAD  ...
Processing  3720  ...
Processing  5751  ...
Processing  3767  ...
Processing  2799  ...
Processing  6535  ...
Processing  4573  ...
Processing  2353  ...
Processing  3355  ...
Processing  4000  ...
Processing  1353  ...
Processing  5379  ...
Processing  VNREAL  ...
Processing  3530  ...
Processing  2350  ...
Processing  2797  ...
Processing  8350  ...
Processing  0583  ...
Processing  9572  ...
Processing  9574  ...
Processing  5550  ...
Processing  2700  ...
Processing  4535  ...
Processing  3740  ...
Processing  VNDIAMOND  ...
Processing  1777  ...
Processing  8781  ...
Processing  5370  ...
Processing  7000  ...
Processing  7500  ...
Processing  8771  ...
Processing  1757  ...
Processing  8300  ...
Processing  9537  